[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/templates/56_capsule_overlap.ipynb)

# 🟡 Medium: Batch Capsule-Capsule Overlap

A **capsule** is the set of all points within distance `r` of a line segment (its *spine*). Given `N` capsules `caps_a` and `M` capsules `caps_b`, return an `(N, M)` boolean array where `out[i, j]` is `True` if capsule `i` and capsule `j` overlap.

Each capsule is `[x1, y1, x2, y2, r]` where `(x1,y1)→(x2,y2)` is the spine and `r` is the radius.

### Signature
```python
def capsule_overlap(caps_a: np.ndarray, caps_b: np.ndarray) -> np.ndarray:
    # caps_a: (N, 5) float — [x1, y1, x2, y2, r]
    # caps_b: (M, 5) float
    # returns: (N, M) bool
```

### Rules
- Do **NOT** use Python `for` loops over `N` or `M`
- Two capsules touching exactly (`dist == r1+r2`) count as overlapping
- A capsule with a zero-length spine is a circle

### Example
```
caps_a = [[0, 0, 2, 0, 1.0]]    # horizontal spine y=0, radius 1
caps_b = [[0, 1.5, 2, 1.5, 1.0], # spine y=1.5 — touching (dist=1.5 < 2)
           [0, 5.0, 2, 5.0, 0.5]] # spine y=5   — separated (dist=5 > 1.5)
output: [[True, False]]
```

> **Reduction step (say this before coding):** `out[i,j]` is True iff the minimum distance between the spine of `caps_a[i]` and the spine of `caps_b[j]` is less than or equal to `caps_a[i,4] + caps_b[j,4]` — computed by minimising `||P1+s·d1 − P3−t·d2||` over `s,t ∈ [0,1]` with parametric clamping, broadcast over `caps_a[:,None,:]` vs `caps_b[None,:,:]`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def capsule_overlap(caps_a, caps_b):
    # caps_a: (N, 5) — each row [x1, y1, x2, y2, r]
    # caps_b: (M, 5)
    # returns: (N, M) bool
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
caps_a = np.array([[0.0, 0.0, 2.0, 0.0, 1.0]])
caps_b = np.array([[0.0, 1.5, 2.0, 1.5, 1.0],
                   [0.0, 5.0, 2.0, 5.0, 0.5]])
result = capsule_overlap(caps_a, caps_b)
print("shape:", result.shape)   # expect (1, 2)
print("result:", result)         # expect [[True, False]]

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: touching capsules (dist == r_sum → True) ──────────────────────
a = np.array([[0.0, 0.0, 2.0, 0.0, 1.5]])
b = np.array([[0.0, 3.0, 2.0, 3.0, 1.5]])
r = capsule_overlap(a, b)
assert r.shape == (1, 1), f"Shape: {r.shape}"
assert r[0, 0] == True, "Touching capsules should overlap"
print("Test 1 passed: touching capsules")

# ── Test 2: separated capsules ────────────────────────────────────────────
a2 = np.array([[0.0, 0.0, 2.0, 0.0, 0.5]])
b2 = np.array([[0.0, 5.0, 2.0, 5.0, 0.5]])
r2 = capsule_overlap(a2, b2)
assert r2[0, 0] == False, "Separated capsules should not overlap"
print("Test 2 passed: separated capsules")

# ── Test 3: perpendicular crossing spines ─────────────────────────────────
a3 = np.array([[0.0, 1.0, 3.0, 1.0, 0.1]])
b3 = np.array([[1.0, 0.0, 1.0, 3.0, 0.1]])
r3 = capsule_overlap(a3, b3)
assert r3[0, 0] == True, "Crossing spines → overlap"
print("Test 3 passed: perpendicular crossing capsules")

# ── Test 4: degenerate capsules (circles) ─────────────────────────────────
a4 = np.array([[0.0, 0.0, 0.0, 0.0, 2.0]])
b4 = np.array([[1.0, 0.0, 1.0, 0.0, 2.0]])
r4 = capsule_overlap(a4, b4)
assert r4[0, 0] == True, "Overlapping circles"
c4 = np.array([[10.0, 0.0, 10.0, 0.0, 1.0]])
r4b = capsule_overlap(a4, c4)
assert r4b[0, 0] == False, "Separated circles"
print("Test 4 passed: degenerate capsules (circles)")

# ── Test 5: large N=M=500 ─────────────────────────────────────────────────
rng = np.random.default_rng(42)
pts_a = rng.uniform(-50, 50, (500, 4))
pts_b = rng.uniform(-50, 50, (500, 4))
caps_a5 = np.hstack([pts_a, rng.uniform(0.5, 3.0, (500, 1))])
caps_b5 = np.hstack([pts_b, rng.uniform(0.5, 3.0, (500, 1))])
t0 = time.time()
r5 = capsule_overlap(caps_a5, caps_b5)
elapsed = time.time() - t0
assert r5.shape == (500, 500), f"Shape: {r5.shape}"
assert elapsed < 3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: N=M=500 ({elapsed:.3f}s)")

print("\nAll tests passed!")